# Downstream task template: symmetry-aware inputs without paper policy

This notebook is intentionally generic. It shows how to prepare canonical data, optionally materialize a translation orbit, fit/validate a generator, and hand trajectories to a downstream sparse-discovery backend. It does not encode any manuscript-specific branch policy.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import importlib.util
import numpy as np

from notebooks._tutorial_utils import confidence_card, print_cards, pretty_json
from pdelie.data import generate_heat_1d_field_batch, split_batch_train_heldout
from pdelie.discovery import evaluate_discovery_recovery, fit_pysindy_discovery, to_pysindy_trajectories
from pdelie.invariants import build_uniform_translation_orbit_batch
from pdelie.reporting import summarize_generator_fit_diagnostics, summarize_verification_report
from pdelie.residuals import HeatResidualEvaluator
from pdelie.symmetry import fit_translation_generator, validate_symmetry_candidate
from pdelie.verification import verify_translation_generator

CONFIG = {
    "fit_epsilon": 1e-4,
    "orbit_shifts": [0.0, np.pi / 8.0, -np.pi / 8.0],
    "use_orbit_batch": True,
}
CONFIG


{'fit_epsilon': 0.0001,
 'orbit_shifts': [0.0, 0.39269908169872414, -0.39269908169872414],
 'use_orbit_batch': True}

## 1. Create a train/heldout split before optional orbit materialization

The orbit helper records source/shift provenance, but it does not choose split policy. Split first when leakage matters.


In [2]:
field = generate_heat_1d_field_batch(batch_size=6, num_times=17, num_points=32, seed=680)
train, heldout = split_batch_train_heldout(field, train_size=3, seed=681)
if CONFIG["use_orbit_batch"]:
    orbit = build_uniform_translation_orbit_batch(
        train,
        shifts=CONFIG["orbit_shifts"],
        source_field_id="heat_train_seed_680_split_681",
    )
    downstream_field = orbit.field
    orbit_report = orbit.report
else:
    downstream_field = train
    orbit_report = None
print(pretty_json({
    "train_shape": list(train.values.shape),
    "downstream_shape": list(downstream_field.values.shape),
    "orbit_report_type": None if orbit_report is None else orbit_report["summary_type"],
    "leakage_policy": "caller-owned; split before materialization in this template",
}))


{
  "downstream_shape": [
    9,
    17,
    32,
    1
  ],
  "leakage_policy": "caller-owned; split before materialization in this template",
  "orbit_report_type": "uniform_translation_orbit_batch",
  "train_shape": [
    3,
    17,
    32,
    1
  ]
}


## 2. Fit and validate the generator used by the workflow


In [3]:
evaluator = HeatResidualEvaluator()
generator = fit_translation_generator(downstream_field, evaluator, epsilon=CONFIG["fit_epsilon"])
verification = verify_translation_generator(heldout, generator, evaluator)
validation = validate_symmetry_candidate(
    heldout,
    generator,
    residual_evaluator=evaluator,
    source_candidate_id="downstream_template_generator",
)
fit_summary = summarize_generator_fit_diagnostics(generator)
verification_summary = summarize_verification_report(verification)
card = confidence_card(
    label="downstream template generator",
    fit=fit_summary,
    verification=verification_summary,
    validation=validation,
)
print_cards([card])


[
  {
    "candidate_kind": "generator_family",
    "condition_number": 108587.97456358546,
    "evidence_label": "direct_svd_in_tolerance",
    "first_epsilon": 0.0001,
    "first_error": 1.3904654612463122e-08,
    "fit_mode": "svd",
    "label": "downstream template generator",
    "max_error": 1.385275387507436e-05,
    "reference_fallback_used": false,
    "selected_span_distance": 2.5012451564009825e-05,
    "singular_value_count": 4,
    "svd_span_distance": 2.5012451564009825e-05,
    "validation_conclusion": "validated",
    "verification_classification": "exact"
  }
]


## 3. Build backend-native trajectories

`to_pysindy_trajectories(...)` is a narrow bridge format. Its output is not a PDELie canonical object.


In [4]:
trajectories, time_values, feature_names = to_pysindy_trajectories(downstream_field)
print(pretty_json({
    "num_trajectories": len(trajectories),
    "trajectory_shape": list(trajectories[0].shape),
    "num_feature_names": len(feature_names),
}))


{
  "num_feature_names": 32,
  "num_trajectories": 9,
  "trajectory_shape": [
    17,
    32
  ]
}


## 4. Optional PySINDy smoke fit

If PySINDy is installed, run the backend adapter. Either way, keep recovery metrics separate from generator-confidence metrics.


In [5]:
if importlib.util.find_spec("pysindy") is None:
    discovery = {"status": "skipped", "reason": "pysindy is not installed"}
else:
    discovery = fit_pysindy_discovery(trajectories, time_values, feature_names)

# Tiny paper-agnostic recovery-metric example over caller-supplied canonical terms.
recovery = evaluate_discovery_recovery(
    target_terms={"u_xx": 1.0},
    discovered_terms={"u_xx": 0.98, "u": 0.01},
    support_epsilon=0.05,
)
print(pretty_json({
    "discovery_status": discovery["status"],
    "recovery_classification": recovery["classification"],
    "support_f1": recovery["support_f1"],
}, max_chars=2500))


{
  "discovery_status": "success",
  "recovery_classification": "exact",
  "support_f1": 1.0
}


## Takeaway

The reusable pattern is: split policy first, optional orbit materialization with provenance, generator confidence report, then downstream backend-specific work.
